In [1]:
import scanpy as sc 
import os
import numpy as np

In [4]:
scdata = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/grn_momics/multi_rna_raw.h5ad')

new_idx = []
sample = []
for ele in scdata.obs.index.values:
    a,b,c = ele.split('-')
    new_idx.append(f'{b}-{c}-{a}')
    sample.append(a)

scdata.obs.index = new_idx
scdata.obs.loc[:,'sample'] = sample
scdata.obs.head()
scdata

AnnData object with n_obs × n_vars = 21723 × 9911
    obs: 'leiden', 'eco', 'species', 'sample'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'

In [3]:
import pickle

with open('/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_noqc_outs/leiden_cistopic/0_cistopic_obj.pkl', 'rb') as f:
    cisobj = pickle.load(f)

#cisobj.cell_data.head()

idx_1 = scdata.obs.index 
idx_2 = cisobj.cell_data.index
idx = np.intersect1d(idx_1, idx_2)
print(idx.shape)


rna = scdata[idx,:]

/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-16 08:39:38,618	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


(5831,)


In [5]:
rna = scdata
rna.raw = rna 

sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, n_top_genes=3000)
rna = rna[:, rna.var.highly_variable]
sc.pp.scale(rna, max_value=10)
rna.write_h5ad('/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_hm_outs/rna.h5ad')

/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Revieved a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [6]:
rna

AnnData object with n_obs × n_vars = 21723 × 3000
    obs: 'leiden', 'eco', 'species', 'sample'
    var: 'gene_ids', 'feature_types', 'genome', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'log1p', 'hvg'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'

## leiden 1-10

In [21]:
from tqdm import tqdm
for i in tqdm(range(1,11)):
    dir_name = f'leiden_grn/leiden_{i}_pipeline'
    scdata = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/grn_momics/multi_rna_raw.h5ad')

    new_idx = []
    sample = []
    for ele in scdata.obs.index.values:
        a,b,c = ele.split('-')
        new_idx.append(f'{b}-{c}-{a}')
        sample.append(a)

    scdata.obs.index = new_idx
    scdata.obs.loc[:,'sample'] = sample
    scdata.obs.head()
    
    rna = scdata
    rna.raw = rna 

    sc.pp.normalize_total(rna, target_sum=1e4)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(rna, n_top_genes=3000)
    rna = rna[:, rna.var.highly_variable]
    sc.pp.scale(rna, max_value=10)
    rna.write_h5ad(os.path.join(dir_name, 'rna.h5ad'))

  0%|          | 0/10 [00:00<?, ?it/s]/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Revieved a view of an AnnData. Making a copy.
  view_to_actual(adata)
... storing 'sample' as categorical
 10%|█         | 1/10 [00:04<00:41,  4.62s/it]/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Revieved a view of an AnnData. Making a copy.
  view_to_actual(adata)
... storing 'sample' as categorical
 20%|██        | 2/10 [00:09<00:37,  4.63s/it]/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Revieved a view of an AnnData. Making a copy.
  view_to_actual(adata)
... storing 'sample' as categorical
 30%|███       | 3/10 [00:14<00:33,  4.81s/it]/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/scanpy/preprocessing/_simple

## bug fix

In [30]:
import pyranges as pr


path = '/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_noqc_outs/4_outs/region_sets/DARs_cell_type'
f_list = os.listdir(path)

for ele in f_list:

        pr.read_bed(os.path.join(path,ele),
                as_df=False
        )

IndexError: list index out of range

In [31]:
ele

'QF.bed'

In [29]:
path = '/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_noqc_outs/4_outs/region_sets/DARs_cell_type'
os.listdir(path)

['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']

In [26]:
for i in range(11):
    dir = f'/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_noqc_outs/{i}_outs/region_sets/DARs_cell_type'
    print(str(i) + '==========================')
    print(os.listdir(dir))
    

0==========================
['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']
1==========================
['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']
2==========================
['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']
3==========================
['T.bed', 'QF.bed', 'JT.bed', 'M.bed']
4==========================
['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']
5==========================
['T.bed', 'QF.bed', 'M.bed']
6==========================
['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']
7==========================
['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']
8==========================
['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']
9==========================
['T.bed', 'QF.bed', 'CY.bed', 'JT.bed', 'M.bed']
10==========================
['T.bed', 'QF.bed', 'M.bed']


## NOT USED

In [5]:
rna = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/grn_momics/leiden_0_pipeline/rna.h5ad')
rna

AnnData object with n_obs × n_vars = 5831 × 3000
    obs: 'leiden', 'eco', 'species', 'sample'
    var: 'gene_ids', 'feature_types', 'genome', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'log1p'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'

In [7]:
import scenicplus
import sys 
sys.path.append('/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/scenicplus/data_wrangling')
from adata_cistopic_wrangling import process_multiome_data
import logging
import random
import sys
from random import sample
from typing import Any, Callable, Dict, Union

import numpy as np
import pandas as pd
from mudata import AnnData, MuData
from pycisTopic.cistopic_class import CistopicObject
from pycisTopic.diff_features import impute_accessibility

from scenicplus.utils import Groupby

In [50]:
process_multiome_data( GEX_anndata = rna,
        cisTopic_obj = cisobj,
        use_raw_for_GEX_anndata = True,
        imputed_acc_kwargs = None,
        bc_transform_func =  lambda x: f'{x}')

2025-06-16 07:57:58,962 Ingesting multiome data INFO     Found 5831 multiome cells.


2025-06-16 07:57:59,157 cisTopic     INFO     Imputing region accessibility
2025-06-16 07:57:59,158 cisTopic     INFO     Impute region accessibility for regions 0-20000
2025-06-16 07:57:59,672 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2025-06-16 07:58:00,181 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2025-06-16 07:58:00,694 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2025-06-16 07:58:01,536 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2025-06-16 07:58:02,048 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2025-06-16 07:58:02,561 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2025-06-16 07:58:03,094 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2025-06-16 07:58:03,612 cisTopic     INFO     Impute region accessibility for regions 160000-180000
2025-06-16 07:58:04,119 cisTopic     

ValueError: Observations annot. `obs` must have as many rows as `X` has rows (5957), but has 5849 rows.

In [8]:
GEX_anndata = rna,
GEX_anndata = GEX_anndata[0]
cisTopic_obj = cisobj,
cisTopic_obj = cisTopic_obj[0]
use_raw_for_GEX_anndata = True,
imputed_acc_kwargs = None,
bc_transform_func =  lambda x: f'{x}'
import logging

In [9]:
cisTopic_obj

In [10]:
# Create logger
level = logging.INFO
format = "%(asctime)s %(name)-12s %(levelname)-8s %(message)s"
handlers = [logging.StreamHandler(stream=sys.stdout)]
logging.basicConfig(level=level, format=format, handlers=handlers)
log = logging.getLogger("Ingesting multiome data")

if imputed_acc_kwargs is None:
    imputed_acc_kwargs = {"scale_factor": 10**6}

# TODO: .raw is deprecated, use layers instead.
GEX_anndata = GEX_anndata.raw.to_adata() if use_raw_for_GEX_anndata else GEX_anndata

# Use bc_transform_func to map scRNA-seq barcodes to scATAC-seq barcodes
GEX_cell_names = [
    bc_transform_func(bc) for bc in GEX_anndata.obs_names]
GEX_anndata.obs_names = GEX_cell_names
ACC_cell_names = list(cisTopic_obj.cell_names.copy())

# get cells with high quality (HQ cells) chromatin accessbility
# AND gene expression profile
common_cells = list(set(GEX_cell_names) & set(ACC_cell_names))
if len(common_cells) == 0:
    raise Exception(
        "No cells found which are present in both assays, check input and consider using `bc_transform_func`!")
log.info(f"Found {len(common_cells)} multiome cells.")

2025-06-16 08:40:25,791 Ingesting multiome data INFO     Found 5831 multiome cells.


In [13]:
# Subset and impute accessibility
imputed_acc_obj = impute_accessibility(
    cisTopic_obj, selected_cells=common_cells, **{"scale_factor": 10**6})
# Subset and get gene expression data
X_EXP = GEX_anndata[common_cells].X.copy()

2025-06-16 08:42:11,888 cisTopic     INFO     Imputing region accessibility
2025-06-16 08:42:11,889 cisTopic     INFO     Impute region accessibility for regions 0-20000
2025-06-16 08:42:12,398 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2025-06-16 08:42:12,896 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2025-06-16 08:42:13,391 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2025-06-16 08:42:14,029 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2025-06-16 08:42:14,514 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2025-06-16 08:42:15,005 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2025-06-16 08:42:15,497 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2025-06-16 08:42:15,988 cisTopic     INFO     Impute region accessibility for regions 160000-180000
2025-06-16 08:42:16,478 cisTopic     

In [14]:
assert all(
    rna_bc == atac_bc
    for rna_bc, atac_bc in zip(
    GEX_anndata[common_cells].to_df().index, imputed_acc_obj.cell_names)
), \
        "rna and atac bc don't match"


In [15]:
# Get and subset metadata from anndata and cisTopic object
GEX_gene_metadata = GEX_anndata.var.copy(deep=True)
GEX_cell_metadata = GEX_anndata.obs.copy(deep=True).loc[
    common_cells]
ACC_region_metadata = pd.DataFrame(index = imputed_acc_obj.feature_names)
ACC_region_metadata = ACC_region_metadata.merge(
    cisTopic_obj.region_data, left_index = True, right_index = True, how = "left")
ACC_region_metadata = ACC_region_metadata.loc[imputed_acc_obj.feature_names]
ACC_cell_metadata = cisTopic_obj.cell_data.copy(deep=True).loc[
    common_cells]

In [19]:
df = ACC_cell_metadata
df = df.drop_duplicates(keep='first')
df.shape

(5831, 27)

In [17]:
tmp = []
for ele in ACC_cell_metadata.index:
    if ele not in tmp:
        tmp.append(ele)
    else:
        print(ele)
        break

ATGAATGCAAGGTACG-1-THM


In [18]:

ACC_cell_metadata.loc['ATGAATGCAAGGTACG-1-THM',:]

,cisTopic_nr_frag,cisTopic_log_nr_frag,cisTopic_nr_acc,cisTopic_log_nr_acc,sample_id,barcode_rank,total_fragments_count,log10_total_fragments_count,unique_fragments_count,log10_unique_fragments_count,...,nucleosome_signal,tss_enrichment,pdf_values_for_tss_enrichment,pdf_values_for_fraction_of_fragments_in_peaks,pdf_values_for_duplication_ratio,barcode,leiden,eco,species,sample
ATGAATGCAAGGTACG-1-THM,7810,3.892651,7803,3.892262,THM,2224,18943,4.277472,8772,3.943148,...,1.505882,1.228856,2.989483,53.451076,66.118563,ATGAATGCAAGGTACG-1,0,eco,T,THM
ATGAATGCAAGGTACG-1-THM,7810,3.892651,7803,3.892262,THM,2224,18943,4.277472,8772,3.943148,...,1.505882,1.228856,2.989483,53.451076,66.118563,ATGAATGCAAGGTACG-1,0,eco,T,THM


In [ ]:
cisTopic_obj.cell_data.copy(deep=True).loc[
    common_cells]

In [83]:
# get cell dimensionality reductions
GEX_dr_cell = {
    key: pd.DataFrame(
        index = GEX_cell_names,
        data=GEX_anndata.obsm[key]).loc[common_cells].to_numpy()
    for key in GEX_anndata.obsm}
ACC_dr_cell = {
    key: cisTopic_obj.projections["cell"][key].loc[common_cells].to_numpy()
    for key in cisTopic_obj.projections["cell"]}

In [110]:
cell_topic.shape


(45, 5957)

In [101]:
ACC_cell_metadata.infer_objects().shape
cistopic_obj = cisobj

In [120]:
model = cistopic_obj.selected_model
cell_names = cistopic_obj.cell_names
cell_topic = model.cell_topic.loc[:, cell_names]
# 使用 ~df.columns.duplicated() 来筛选出第一次出现的列
cell_topic = cell_topic.loc[:, ~cell_topic.columns.duplicated()]
region_names = cistopic_obj.region_names
topic_region = model.topic_region.loc[region_names]
if selected_cells is not None:
    cell_topic = cell_topic.loc[:, selected_cells]
    cell_names = selected_cells
if selected_regions is not None:
    topic_region = topic_region.loc[selected_regions]
    region_names = selected_regions
# Convert cell_topic and topic_region 2d arrays to np.float32 so
# multiplying them uses 4 times less memory than with np.float64
cell_topic = cell_topic.to_numpy().astype(np.float32)
topic_region = topic_region.to_numpy().astype(np.float32)

In [123]:
cell_topic.shape

(45, 5831)

In [12]:
ACC_cell_metadata.infer_objects()

NameError: name 'ACC_cell_metadata' is not defined

In [84]:
DIC =     {
        "scRNA": AnnData(
            X=X_EXP, obs=GEX_cell_metadata.infer_objects(),
            var=GEX_gene_metadata.infer_objects(), obsm=GEX_dr_cell),
        "scATAC": AnnData(
            X=imputed_acc_obj.mtx.T, obs=ACC_cell_metadata.infer_objects(),
            var=ACC_region_metadata.infer_objects(), obsm=ACC_dr_cell)}

ValueError: Observations annot. `obs` must have as many rows as `X` has rows (5957), but has 5849 rows.

In [ ]:
# Subset and impute accessibility
imputed_acc_obj = impute_accessibility(
    cisTopic_obj, selected_cells=common_cells, **imputed_acc_kwargs)
# Subset and get gene expression data
X_EXP = GEX_anndata[common_cells].X.copy()

assert all(
    rna_bc == atac_bc
    for rna_bc, atac_bc in zip(
    GEX_anndata[common_cells].to_df().index, imputed_acc_obj.cell_names)
), \
        "rna and atac bc don't match"

# Get and subset metadata from anndata and cisTopic object
GEX_gene_metadata = GEX_anndata.var.copy(deep=True)
GEX_cell_metadata = GEX_anndata.obs.copy(deep=True).loc[
    common_cells]
ACC_region_metadata = pd.DataFrame(index = imputed_acc_obj.feature_names)
ACC_region_metadata = ACC_region_metadata.merge(
    cisTopic_obj.region_data, left_index = True, right_index = True, how = "left")
ACC_region_metadata = ACC_region_metadata.loc[imputed_acc_obj.feature_names]
ACC_cell_metadata = cisTopic_obj.cell_data.copy(deep=True).loc[
    common_cells]

# get cell dimensionality reductions
GEX_dr_cell = {
    key: pd.DataFrame(
        index = GEX_cell_names,
        data=GEX_anndata.obsm[key]).loc[common_cells].to_numpy()
    for key in GEX_anndata.obsm}
ACC_dr_cell = {
    key: cisTopic_obj.projections["cell"][key].loc[common_cells].to_numpy()
    for key in cisTopic_obj.projections["cell"]}

mudata = MuData(
    {
        "scRNA": AnnData(
            X=X_EXP, obs=GEX_cell_metadata.infer_objects(),
            var=GEX_gene_metadata.infer_objects(), obsm=GEX_dr_cell),
        "scATAC": AnnData(
            X=imputed_acc_obj.mtx.T, obs=ACC_cell_metadata.infer_objects(),
            var=ACC_region_metadata.infer_objects(), obsm=ACC_dr_cell)}
)